In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pandapower.timeseries.data_sources.frame_data import DFData
from pandaprosumer.create import create_empty_prosumer_container, create_period
from pandaprosumer.create_controlled import (create_controlled_const_profile, create_controlled_pv_production)
from pandaprosumer.mapping import GenericMapping
from pandaprosumer.run_time_series import run_timeseries

from pandaprosumer.data_sources.pv_production import PvProduction

In [2]:
inputs = {
    "latitude": 40,
    "longitude": 0,
    "surface_tilt": 40,
    "surface_azimuth": 0,
    "peakpower": 1,          # [kW]
    "loss": 0,               # %
    "raddatabase": "PVGIS-ERA5",
    "start": 2012,
    "end": 2013,
    "url": "https://re.jrc.ec.europa.eu/api/v5_2/seriescalc?",
}

pv_prod = PvProduction(inputs)
pv_prod.calculate_pv()

pv_prod.output[0]

,p_w,poa_direct_w_m2,poa_sky_diffuse_w_m2,poa_ground_diffuse_w_m2,solar_elevation_deg,temp_air_degC,wind_speed_m_s,solar_rad_reconstr_bool
time,,,,,,,,
2012-01-01 00:30:00+00:00,0.0,0.0,0.0,0.0,0.0,11.37,2.14,0
2012-01-01 01:30:00+00:00,0.0,0.0,0.0,0.0,0.0,11.20,2.00,0
2012-01-01 02:30:00+00:00,0.0,0.0,0.0,0.0,0.0,9.79,2.14,0
2012-01-01 03:30:00+00:00,0.0,0.0,0.0,0.0,0.0,8.31,2.41,0
2012-01-01 04:30:00+00:00,0.0,0.0,0.0,0.0,0.0,8.46,2.62,0
...,...,...,...,...,...,...,...,...
2013-12-31 19:30:00+00:00,0.0,0.0,0.0,0.0,0.0,10.82,0.83,0
2013-12-31 20:30:00+00:00,0.0,0.0,0.0,0.0,0.0,8.56,1.45,0
2013-12-31 21:30:00+00:00,0.0,0.0,0.0,0.0,0.0,7.38,1.86,0


In [3]:
start = '2005-01-01 00:30:00'
end   = '2005-01-05 00:29:59'
time_resolution_s = 3600
frequency = '60min'

In [4]:
inputs = {
    "latitude": 40,
    "longitude": 0,
    "surface_tilt": 40,
    "surface_azimuth": 0,
    "peakpower": 5,     # kW
    "loss": 10,         # %
    "raddatabase": "PVGIS-ERA5",
    "start": 2005,
    "end": 2006,
    "url": "https://re.jrc.ec.europa.eu/api/v5_2/seriescalc?"
}

pv_prod = PvProduction(inputs)
pv_prod.calculate_pv()

pv_full = pv_prod.output[0]

pv_df = pv_full.loc[start:end].copy()

if "temp_air_degC" in pv_df.columns:
    pv_df = pv_df.rename(columns={"temp_air_degC":"temp_air_c"})

In [5]:
# simple example heat demand [kW]
n = len(pv_df)
pv_df["q_demand_kw"] = 4.0 + 2.0*np.sin(np.linspace(0, 2*np.pi, n))
data_source = DFData(pv_df)

In [6]:
prosumer = create_empty_prosumer_container()

period = create_period(
    prosumer,
    time_resolution_s,
    start,
    end,
    timezone="utc",
    name="default",
)

In [7]:
input_params = [
    "p_w",
    "poa_direct_w_m2",
    "poa_sky_diffuse_w_m2",
    "poa_ground_diffuse_w_m2",
    "solar_elevation_deg",
    "temp_air_c",
    "wind_speed_m_s",
    "solar_rad_reconstr_bool",
    "q_demand_kw",
]

result_params = [
    "p_w_cp",
    "poa_direct_w_m2_cp",
    "poa_sky_diffuse_w_m2_cp",
    "poa_ground_diffuse_w_m2_cp",
    "solar_elevation_deg_cp",
    "temp_air_c_cp",
    "wind_speed_m_s_cp",
    "solar_rad_reconstr_bool_cp",
    "q_demand_kw_cp",
]

cp_index = create_controlled_const_profile(
    prosumer,
    input_params,
    result_params,
    data_source,
    period=period,
)

In [ ]:
pv_params = dict(
    latitude=inputs["latitude"],
    longitude=inputs["longitude"],
    peakpower=inputs["peakpower"],
    loss=inputs["loss"]
)

pv_index = create_controlled_pv_production(
    prosumer,
    name="pv_rooftop",
    level=1,
    order=0,
    period=period,
    **pv_params
)

In [9]:
from pandaprosumer.create_controlled import create_controlled_heat_demand

hd_index = create_controlled_heat_demand(
    prosumer,
    level=1,
    order=1,
    t_feed_demand_c=30,
    t_return_demand_c=25,
)

In [10]:
# CP → PV
GenericMapping(
    prosumer,
    initiator_id=cp_index,
    initiator_column=[
        "p_w_cp",
        "poa_direct_w_m2_cp",
        "poa_sky_diffuse_w_m2_cp",
        "poa_ground_diffuse_w_m2_cp",
        "solar_elevation_deg_cp",
        "temp_air_c_cp",
        "wind_speed_m_s_cp",
        "solar_rad_reconstr_bool_cp",
    ],
    responder_id=pv_index,
    responder_column=[
        "p_w",
        "poa_direct_w_m2",
        "poa_sky_diffuse_w_m2",
        "poa_ground_diffuse_w_m2",
        "solar_elevation_deg",
        "temp_air_c",
        "wind_speed_m_s",
        "solar_rad_reconstr_bool",
    ],
)

# CP → Heat demand
GenericMapping(
    prosumer,
    initiator_id=cp_index,
    initiator_column="q_demand_kw_cp",
    responder_id=hd_index,
    responder_column="q_demand_kw",
)

# PV → Heat demand
GenericMapping(
    prosumer,
    initiator_id=pv_index,
    initiator_column="p_w",
    responder_id=hd_index,
    responder_column="q_received_kw",
)

In [13]:
run_timeseries(prosumer, period)

results_df = prosumer.time_series.data_source.iloc[1].df
results_df.head(20)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 96/96 [00:02<00:00, 36.90it/s]


,q_received_kw,q_uncovered_kw,mdot_kg_per_s,t_in_c,t_out_c
2005-01-01 00:30:00+00:00,0.00,4.000000,0.0,0.0,0.0
2005-01-01 01:30:00+00:00,0.00,4.132181,0.0,0.0,0.0
2005-01-01 02:30:00+00:00,0.00,4.263784,0.0,0.0,0.0
2005-01-01 03:30:00+00:00,0.00,4.394234,0.0,0.0,0.0
2005-01-01 04:30:00+00:00,0.00,4.522960,0.0,0.0,0.0
2005-01-01 05:30:00+00:00,0.00,4.649399,0.0,0.0,0.0
2005-01-01 06:30:00+00:00,0.00,4.772998,0.0,0.0,0.0
2005-01-01 07:30:00+00:00,1.85,3.043218,0.0,0.0,0.0
2005-01-01 08:30:00+00:00,101.30,-96.290469,0.0,0.0,0.0
2005-01-01 09:30:00+00:00,167.10,-161.978570,0.0,0.0,0.0
